# Task 1
## Ingest IBM DB2 Sample XML files

source: https://www.ibm.com/docs/en/i/7.6.0?topic=tables-sample-xml
note: I fetched with help of claude code and saved as the raw XML

In [0]:
# check the uploaded files from IBM sent by David
%ls /Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/

## Customer 

In [0]:
%cat /Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/customer_1000.xml

In [0]:
customer_df = spark.read.format("xml").option("rowTag", "customerinfo").load("/Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/customer_*.xml")
customer_df.show()

In [0]:
from pyspark.sql.functions import col

customer_df = customer_df.select(
    col("_Cid").alias("customer_id"),
    col("name"),
    col("addr").getField("street").alias("address_street"),
    col("addr").getField("city").alias("address_city"),
    col("addr").getField("_country").alias("address_country"),
    col("addr").getField("pcode-zip").alias("address_pcode_zip"),
    col("addr").getField("prov-state").alias("address_prov_state"),
    col("assistant").getField("name").alias("assistant_name"),
    col("phone").alias("phone")
)

customer_df.show(truncate=False)

In [0]:
%sql
--drop table if exists ibm_customers

In [0]:
customer_df.write\
    .mode("overwrite")\
    .saveAsTable("ibm_customers")

In [0]:
%sql
SELECT * FROM ibm_customers

## Product

In [0]:
%cat /Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/product_100-100-01.xml

In [0]:
product_df = spark.read.format("xml").option("rowTag", "product").load("/Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/product_*.xml")
product_df.show()

In [0]:
from pyspark.sql.functions import col

product_df = product_df.select(
    col("_pid").alias("product_id"),
    col("description").getField("name").alias("product_name"),
    col("description").getField("details").alias("product_description"),
    col("description").getField("price").alias("price"),
    col("description").getField("weight").alias("weight")
)

product_df.show(truncate=False)

In [0]:
%sql
--drop table if exists ibm_products

In [0]:
product_df.write\
    .mode("overwrite")\
    .saveAsTable("ibm_products")

In [0]:
%sql
SELECT * FROM ibm_products

## Purchase Order

In [0]:
%cat /Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/purchaseorder_5000.xml

In [0]:
purchaseorder_df = spark.read.format("xml").option("rowTag", "PurchaseOrder").load("/Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/purchaseorder_*.xml")
purchaseorder_df.display()

In [0]:
from pyspark.sql.functions import col, explode

#purchaseorder_df.printSchema()

purchaseorder_items_df = purchaseorder_df.select(
    col("_OrderDate").alias("order_date"),
    col("_PoNum").alias("po_number"),
    col("_Status").alias("status"),
    explode(col("item")).alias("lineitem")
).select(
    col("order_date"),
    col("po_number"),
    col("status"),
    col("lineitem.name").alias("product_name"),
    col("lineitem.partid").alias("part_id"),
    col("lineitem.price").alias("price"),
    col("lineitem.quantity").alias("quantity")
)

purchaseorder_items_df.show(truncate=False)

In [0]:
%sql
--drop table if exists ibm_purchaseorders

In [0]:
purchaseorder_items_df.write\
    .mode("overwrite")\
    .saveAsTable("ibm_purchaseorders")

In [0]:
%sql
SELECT * FROM ibm_purchaseorders

## Supplier

In [0]:
%cat /Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/suppliers_100.xml

In [0]:
supplier_df = spark.read.format("xml").option("rowTag", "supplierinfo").load("/Volumes/workspace/default/raw_files/ibm_db2_sample_xml_files/suppliers_*.xml")
supplier_df.display()

In [0]:
from pyspark.sql.functions import col

supplier_df = supplier_df.select(
    col("_Sid").alias("supplier_id"),
    col("name").alias("supplier_name"),
    col("addr").getField("street").alias("address_street"),
    col("addr").getField("city").alias("address_city"),
    col("addr").getField("_country").alias("address_country"),
    col("addr").getField("pcode-zip").alias("address_pcode_zip"),
    col("addr").getField("prov-state").alias("address_prov_state")
)

supplier_df.show()

In [0]:
%sql
--drop table if exists ibm_suppliers

In [0]:
supplier_df.write\
    .mode("overwrite")\
    .saveAsTable("ibm_suppliers")

In [0]:
%sql
SELECT * FROM ibm_suppliers